# pytorch-mid: Medical segmentation-Informed Divergence

This notebook demonstrates the `pytorch-mid` API using random vectors as stand-ins for MAISI latent features.

**Setup:** `pip install pytorch-mid` (or `uv add pytorch-mid`)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from pytorch_mid import compute_mid_from_latents, PCAProjection
from pytorch_mid.decomposition import compute_mid_decomposed

RNG = np.random.default_rng(42)

## 1. Setup: simulate joint latent vectors

In practice these come from a MAISI encoder applied to (image, mask) pairs.
Here we use random Gaussians to illustrate the API.

Each vector is the concatenation of a flattened image latent and a flattened mask latent,
after spatial pooling to `(4, 8, 8, 4)` per modality → **2 × 1024 = 2048 dims**.

In [ ]:
D = 2048   # joint latent dimension (image + mask halves)
N = 200    # samples per set  (must be > k for PCA)

# Reference: ground-truth (image, mask) pairs ~ N(0, I)
ref = RNG.standard_normal((N, D)).astype(np.float32)

# Fit PCA on reference set
proj = PCAProjection(k=128).fit(ref)
print(f"PCA k=128 explains {proj.variance_explained()*100:.1f}% of variance")

## 2. Scalar MID: ordering by distribution shift

In [ ]:
# Three candidate sets at increasing distance from reference
self_  = ref.copy()                                                    # perfect match
close  = RNG.standard_normal((N, D)).astype(np.float32) + 0.2         # slight shift
medium = RNG.standard_normal((N, D)).astype(np.float32) + 1.5         # moderate shift
far    = RNG.standard_normal((N, D)).astype(np.float32) + 3.0         # large shift

scores = {
    "self (perfect)": compute_mid_from_latents(ref, self_,  projection=proj),
    "close (shift=0.2)": compute_mid_from_latents(ref, close,  projection=proj),
    "medium (shift=1.5)": compute_mid_from_latents(ref, medium, projection=proj),
    "far (shift=3.0)": compute_mid_from_latents(ref, far,    projection=proj),
}

for name, s in scores.items():
    print(f"  MID({name:25s}) = {s:.2f}")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))
ax.bar(scores.keys(), scores.values(), color=["#4878d0", "#6acc65", "#ee854a", "#d65f5f"])
ax.set_ylabel("MID")
ax.set_title("MID increases with distribution shift")
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.show()

## 3. Sensitivity decomposition: image / mask / cross terms

MID decomposes into three interpretable components:

| Term | What it captures |
|------|------------------|
| `MID_img`  | Appearance difference (image latent shift) |
| `MID_mask` | Shape difference (mask latent shift) |
| `MID_cross` | Image-mask alignment (cross-covariance) |

The cross term captures mismatches between image and mask that the individual terms miss
(e.g., a plausible mask placed on the wrong image).

In [ ]:
scenarios = {}

# Scenario A: only mask half shifted (simulates wrong segmentation, correct image)
mask_shifted = ref.copy()
mask_shifted[:, D // 2:] += 3.0
scenarios["A: mask shift only"] = compute_mid_decomposed(
    proj.transform(ref), proj.transform(mask_shifted)
)

# Scenario B: only image half shifted (simulates different scanner, same anatomy)
img_shifted = ref.copy()
img_shifted[:, :D // 2] += 3.0
scenarios["B: image shift only"] = compute_mid_decomposed(
    proj.transform(ref), proj.transform(img_shifted)
)

# Scenario C: shuffled pairs (correct marginals, broken alignment)
shuffled = np.concatenate([
    ref[:, :D // 2],                          # original image halves
    RNG.permutation(ref[:, D // 2:]),          # shuffled mask halves
], axis=1)
scenarios["C: shuffled pairs"] = compute_mid_decomposed(
    proj.transform(ref), proj.transform(shuffled)
)

# Display
print(f"{'Scenario':<25}  {'total':>8}  {'img':>8}  {'mask':>8}  {'cross':>8}")
print("-" * 65)
for name, d in scenarios.items():
    print(f"{name:<25}  {d['total']:8.2f}  {d['img']:8.2f}  {d['mask']:8.2f}  {d['cross']:8.2f}")

In [ ]:
terms = ["img", "mask", "cross"]
colors = ["#4878d0", "#ee854a", "#6acc65"]
x = np.arange(len(scenarios))
width = 0.25

fig, ax = plt.subplots(figsize=(7, 4))
for i, (term, color) in enumerate(zip(terms, colors)):
    vals = [d[term] for d in scenarios.values()]
    ax.bar(x + i * width, vals, width, label=f"MID_{term}", color=color, alpha=0.85)

ax.set_xticks(x + width)
ax.set_xticklabels(scenarios.keys(), rotation=10, ha="right")
ax.set_ylabel("MID component")
ax.set_title("Decomposition reveals failure mode")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

**Interpretation:**
- Scenario A (mask shift): `MID_mask` dominates — the mask distribution has changed.
- Scenario B (image shift): `MID_img` dominates — the image distribution has changed.
- Scenario C (shuffled pairs): `MID_cross` spikes — image-mask alignment is broken even though marginals are identical.

## 4. Monotonicity with corruption severity

Simulate the severity-sweep experiment: apply increasing corruption to the mask half and show MID tracks it.

In [ ]:
severities = np.linspace(0, 4, 20)
mid_vals, mask_vals, img_vals = [], [], []

for s in severities:
    cand = ref.copy()
    cand[:, D // 2:] += s     # progressive mask shift (simulates growing corruption)
    d = compute_mid_decomposed(proj.transform(ref), proj.transform(cand))
    mid_vals.append(d["total"])
    mask_vals.append(d["mask"])
    img_vals.append(d["img"])

# Proxy Dice: 1/(1 + shift) as a simple quality proxy
proxy_dice = 1.0 / (1.0 + severities)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

ax = axes[0]
ax.plot(severities, mid_vals,  label="MID (joint)",  color="#4878d0")
ax.plot(severities, mask_vals, label="MID_mask",      color="#ee854a", linestyle="--")
ax.plot(severities, img_vals,  label="MID_img",       color="#6acc65", linestyle=":")
ax.set_xlabel("Corruption severity")
ax.set_ylabel("MID")
ax.set_title("MID vs severity")
ax.legend(frameon=False)
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(proxy_dice, mid_vals, color="#4878d0", marker="o", markersize=3)
ax.set_xlabel("Quality proxy (1 / (1 + severity))")
ax.set_ylabel("MID")
ax.set_title("MID vs quality (invert x-axis = degrading quality)")
ax.invert_xaxis()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

from scipy.stats import spearmanr
rho, pval = spearmanr(mid_vals, -np.array(proxy_dice))
print(f"Spearman rho (MID vs -quality): {rho:.3f}  (p={pval:.2e})")

## 5. Sample complexity: how many samples are needed?

MID stabilises with fewer samples than FID on natural images because the MAISI latent space is more compact and structured.

In [ ]:
ns = [10, 25, 50, 100, 200, 500]
n_repeats = 20
true_score = compute_mid_from_latents(ref, far, projection=proj)

estimates = {n: [] for n in ns}
for n in ns:
    for _ in range(n_repeats):
        idx_r = RNG.choice(len(ref), n, replace=False)
        idx_f = RNG.choice(len(far), n, replace=False)
        try:
            s = compute_mid_from_latents(ref[idx_r], far[idx_f], projection=proj)
            estimates[n].append(s)
        except Exception:
            pass

means = [np.mean(estimates[n]) for n in ns]
stds  = [np.std(estimates[n])  for n in ns]
cvs   = [s / m * 100 for m, s in zip(means, stds)]

print(f"True MID (N={len(ref)}): {true_score:.2f}")
print(f"{'n':>6}  {'mean':>8}  {'std':>8}  {'CV %':>8}")
for n, m, s, cv in zip(ns, means, stds, cvs):
    print(f"{n:6d}  {m:8.2f}  {s:8.2f}  {cv:8.1f}%")

fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(ns, cvs, "o-", color="#4878d0")
ax.axhline(10, color="grey", linestyle="--", linewidth=0.8, label="CV = 10%")
ax.set_xscale("log")
ax.set_xlabel("Number of samples $n$")
ax.set_ylabel("Coefficient of variation (%)")
ax.set_title("Sample complexity of MID estimator")
ax.legend(frameon=False)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Summary

| Feature | pytorch-mid |
|---------|-------------|
| Scalar quality metric | `compute_mid_from_latents(ref, cand, proj)` |
| Interpretable decomposition | `compute_mid_decomposed(ref_proj, cand_proj)` |
| Tracks corruption severity | Spearman ρ ≈ −1 in this demo |
| Detects alignment errors | Cross term spikes on shuffled pairs |
| No per-sample GT pairing | Distributional metric — only sets needed |

To use with real 3D medical volumes, supply a MAISI encoder instance — see the README.